# Residual по инверсии BGG Rank

Отдельный прогон идей, которых нет в `kaggle.ipynb`.

База — обратная формула geek rating. Ранг задаёт порядок `G`, не само `G`.
Дальше остаток учат деревья.

Сравниваются шесть прогнозов на одном 5-fold:

1. инверсия (изотоника, сплайн по бинам или `C = a + b·log(n)` / `sqrt(n)`, что лучше на сетке)
2. stage1: LightGBM только по рангу и числу оценок
3. инверсия + LightGBM Huber на остатке без ранг-фич, 15 листьев
4. stage1 + LightGBM на остатке
5. одна модель на всех фичах, куда инверсия, stage1 и мелкое дерево (глубина 3) добавлены колонками
6. смесь inv_residual и full: вес `n/(n+100)` в сторону остатка


In [1]:
import os
import numpy as np
import pandas as pd

N_JOBS = 2
os.environ['OMP_NUM_THREADS'] = str(N_JOBS)
os.environ['MKL_NUM_THREADS'] = str(N_JOBS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_JOBS)

def comma_to_float(s):
    return pd.to_numeric(
        pd.Series(s, copy=False).astype(str).str.replace(',', '.', regex=False),
        errors='coerce',
    )

def load_bgg(path):
    d = pd.read_csv(path)
    if 'Rating Average' in d.columns:
        d['Rating Average'] = comma_to_float(d['Rating Average'])
    d['Complexity Average'] = comma_to_float(d['Complexity Average'])
    d['Mechanics'] = d['Mechanics'].fillna('Unknown')
    d['Domains'] = d['Domains'].fillna('Unknown')
    return d

train = load_bgg('train_data.csv')
test = load_bgg('test_data.csv')
print(train.shape, test.shape)


(15257, 14) (5086, 13)


In [2]:
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

y = train['Rating Average'].to_numpy(float)
n_votes = train['Users Rated'].to_numpy(float)
rank = train['BGG Rank'].to_numpy(float)
n_test = test['Users Rated'].to_numpy(float)
rank_test = test['BGG Rank'].to_numpy(float)


def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))


def C_vec(n, spec):
    n = np.asarray(n, dtype=float)
    kind = spec['kind']
    if kind in ('isotonic', 'spline', 'pairc'):
        return np.full(len(n), float(spec['C']))
    if kind == 'clog':
        return np.clip(spec['a'] + spec['b'] * np.log1p(n), 80.0, None)
    if kind == 'csqrt':
        return np.clip(spec['a'] + spec['b'] * np.sqrt(np.clip(n, 0, None)), 80.0, None)
    raise ValueError(kind)


def estimate_pair_C(rank_tr, y_tr, n_tr, prior=5.55):
    order = np.argsort(rank_tr)
    r = rank_tr[order]
    yy = y_tr[order]
    nn = n_tr[order]
    n1, n2 = nn[:-1], nn[1:]
    y1, y2 = yy[:-1], yy[1:]
    ok = (np.abs(r[1:] - r[:-1]) <= 8) & (np.minimum(n1, n2) >= 50) & (np.abs(n1 - n2) >= 30)
    den = n1 * y1 - n2 * y2 + prior * (n2 - n1)
    with np.errstate(divide='ignore', invalid='ignore'):
        C = n1 * n2 * (y2 - y1) / den
    C = C[ok & np.isfinite(C) & (C > 200) & (C < 6000)]
    return float(np.median(C)) if len(C) else 1800.0


def predict_gh_bins(rank_tr, y_tr, n_tr, rank_va, n_va, n_bins=30):
    edges = np.unique(np.quantile(rank_tr, np.linspace(0, 1, n_bins + 1)))
    xs, gs, hs = [], [], []
    for i in range(max(len(edges) - 1, 0)):
        if i == len(edges) - 2:
            mask = rank_tr >= edges[i]
        else:
            mask = (rank_tr >= edges[i]) & (rank_tr < edges[i + 1])
        if int(mask.sum()) < 50:
            continue
        invn = (1.0 / np.maximum(n_tr[mask], 1.0)).reshape(-1, 1)
        est = HuberRegressor(alpha=0.0, max_iter=200)
        est.fit(invn, y_tr[mask])
        xs.append(float(rank_tr[mask].mean()))
        gs.append(float(est.intercept_))
        hs.append(float(est.coef_[0]))
    xs = np.asarray(xs)
    iso_g = IsotonicRegression(increasing=False, out_of_bounds='clip')
    iso_h = IsotonicRegression(increasing=False, out_of_bounds='clip')
    iso_g.fit(xs, gs)
    iso_h.fit(xs, hs)
    return np.clip(iso_g.predict(rank_va) + iso_h.predict(rank_va) / np.maximum(n_va, 1.0), 1, 10)


def predict_gh_win(rank_tr, y_tr, n_tr, rank_va, n_va, k=250):
    order = np.argsort(rank_tr)
    r = rank_tr[order]
    yy = y_tr[order]
    invn = 1.0 / np.maximum(n_tr[order], 1.0)
    ntot = len(r)
    k = int(min(max(k, 40), ntot))
    pos = np.searchsorted(r, rank_va)
    lo = np.clip(pos - k // 2, 0, ntot - k)
    pred = np.empty(len(rank_va))
    eye = np.eye(2) * 1e-3
    for i, start in enumerate(lo):
        sl = slice(start, start + k)
        X = np.column_stack([np.ones(k), invn[sl]])
        xtx = X.T @ X + eye
        coef = np.linalg.solve(xtx, X.T @ yy[sl])
        pred[i] = coef[0] + coef[1] / max(n_va[i], 1.0)
    return np.clip(pred, 1, 10)


def invert_rating(rank_tr, y_tr, n_tr, rank_va, n_va, spec):
    kind = spec['kind']
    if kind == 'gh':
        return predict_gh_bins(rank_tr, y_tr, n_tr, rank_va, n_va, n_bins=spec['bins'])
    if kind == 'gh_win':
        return predict_gh_win(rank_tr, y_tr, n_tr, rank_va, n_va, k=spec['k'])
    if kind == 'pairc':
        spec = dict(kind='isotonic', C=estimate_pair_C(rank_tr, y_tr, n_tr, spec['prior']), prior=spec['prior'])
    C_tr = C_vec(n_tr, spec)
    prior = float(spec['prior'])
    geek = (y_tr * n_tr + C_tr * prior) / (n_tr + C_tr)
    if spec['kind'] == 'spline':
        edges = np.unique(np.quantile(rank_tr, np.linspace(0, 1, 41)))
        bins = np.clip(np.digitize(rank_tr, edges[1:-1]), 0, len(edges) - 2)
        xs, ys = [], []
        for b in range(max(len(edges) - 1, 0)):
            mask = bins == b
            if int(mask.sum()) < 8:
                continue
            xs.append(float(rank_tr[mask].mean()))
            ys.append(float(geek[mask].mean()))
        xs = np.asarray(xs)
        ys = np.asarray(ys)
        order = np.argsort(xs)
        geek_hat = np.interp(rank_va, xs[order], ys[order])
    else:
        iso = IsotonicRegression(increasing=False, out_of_bounds='clip')
        iso.fit(rank_tr, geek)
        geek_hat = iso.predict(rank_va)
    C_va = C_vec(n_va, spec)
    rating = (geek_hat * (n_va + C_va) - C_va * prior) / np.maximum(n_va, 1.0)
    return np.clip(rating, 1, 10)


def inversion_cv(spec, n_splits=3):
    scores = []
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
    for tr, va in kf.split(y):
        pred = invert_rating(rank[tr], y[tr], n_votes[tr], rank[va], n_votes[va], spec)
        scores.append(_rmse(y[va], pred))
    return float(np.mean(scores))


candidates = []
for C in (1600, 2000, 2500):
    for prior in (5.5, 5.55, 5.7):
        candidates.append(dict(kind='isotonic', C=C, prior=prior))
for a in (800, 2000, 3500):
    for b in (-150, 0, 200):
        candidates.append(dict(kind='clog', a=a, b=b, prior=5.55))
for a in (500, 1800):
    for b in (-15, 0, 30):
        candidates.append(dict(kind='csqrt', a=a, b=b, prior=5.55))
for bins in (16, 24, 32, 48):
    candidates.append(dict(kind='gh', bins=bins))
for k in (120, 220, 350, 500):
    candidates.append(dict(kind='gh_win', k=k))
for prior in (5.5, 5.55, 5.6):
    candidates.append(dict(kind='pairc', prior=prior))

scored = [(inversion_cv(spec), spec) for spec in candidates]
scored.sort(key=lambda row: row[0])
best_iso = min((row for row in scored if row[1]['kind'] == 'isotonic'), key=lambda row: row[0])
spline_spec = dict(kind='spline', C=best_iso[1]['C'], prior=best_iso[1]['prior'])
scored.append((inversion_cv(spline_spec), spline_spec))
scored.sort(key=lambda row: row[0])

print('топ инверсий, 3-fold RMSE')
for cv, spec in scored[:8]:
    print(f'  {cv:.4f}  {spec}')
print('data-driven')
for cv, spec in scored:
    if spec['kind'] in ('gh', 'gh_win', 'pairc'):
        print(f'  {cv:.4f}  {spec}')
BEST_SPEC = scored[0][1]
print('BEST_SPEC', BEST_SPEC, 'cv', round(scored[0][0], 4))


топ инверсий, 3-fold RMSE
  0.2424  {'kind': 'csqrt', 'a': 1800, 'b': -15, 'prior': 5.55}
  0.2426  {'kind': 'isotonic', 'C': 2000, 'prior': 5.55}
  0.2426  {'kind': 'clog', 'a': 2000, 'b': 0, 'prior': 5.55}
  0.2428  {'kind': 'csqrt', 'a': 1800, 'b': 0, 'prior': 5.55}
  0.2437  {'kind': 'isotonic', 'C': 1600, 'prior': 5.55}
  0.2439  {'kind': 'isotonic', 'C': 2500, 'prior': 5.55}
  0.2447  {'kind': 'isotonic', 'C': 1600, 'prior': 5.5}
  0.2449  {'kind': 'isotonic', 'C': 2000, 'prior': 5.5}
BEST_SPEC {'kind': 'csqrt', 'a': 1800, 'b': -15, 'prior': 5.55} cv 0.2424


In [3]:
from collections import Counter, defaultdict
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from lightgbm import LGBMRegressor

NUM_RAW = [
    'Year Published', 'Min Players', 'Max Players', 'Play Time',
    'Min Age', 'Users Rated', 'Complexity Average', 'Owned Users', 'BGG Rank',
]
ZERO_AS_NA = ['Year Published', 'Play Time', 'Min Age', 'Complexity Average']
NAME_RE = r'Expanded|Edition|Remastered|Legacy|Deluxe|Reprint|2nd|Second Edition'
GOOD_MECHS = (
    'Variable Player Powers', 'Card Drafting', 'Worker Placement',
    'Deck Bag and Pool Building', 'Cooperative Game', 'Modular Board',
)
TAG_SMOOTH = 80.0
FAMILY_SMOOTH = 8.0
NAME_TFIDF = dict(ngram_range=(1, 2), min_df=5, max_features=3000)
NAME_SVD_COMPS = 16
MECH_SVD_COMPS = 10
TE_COLS = [
    'mean_mech_target', 'max_mech_target', 'min_mech_target', 'count_mech_target', 'var_mech_target',
    'mean_dom_target', 'max_dom_target', 'min_dom_target', 'count_dom_target', 'var_dom_target',
    'family_target', 'family_count',
]
RANK_COLS = [
    'rank', 'log_rank', 'users', 'log_users', 'rank_per_user',
    'vote_weight_1800', 'rank_x_shrink50', 'rank_x_weight1800', 'log_rank_x_weight1800',
]
RANK_DROP = {
    'BGG Rank', 'log_rank', 'rank_per_user',
    'rank_x_shrink', 'rank_x_weight1800', 'log_rank_x_weight1800',
}


def _comma_tokens(text):
    return [t.strip() for t in text.split(',') if t.strip()]


def _norm_tags(s):
    return s.fillna('Unknown').str.replace(r'\s*,\s*', ',', regex=True)


def _mech_doc(text):
    parts = []
    for t in str(text).split(','):
        tok = '_'.join(t.strip().lower().split())
        if tok:
            parts.append(tok)
    return ' '.join(parts)


def _family_keys(names):
    s = pd.Series(names, copy=False).fillna('').astype(str).str.lower()
    s = s.str.replace(r'\([^)]*\)', ' ', regex=True)
    s = s.str.replace(r':.*', ' ', regex=True)
    s = s.str.replace(
        r'\b(\d+(st|nd|rd|th)|second|third|fourth|edition|remastered|deluxe|reprint|expanded|legacy|revised)\b',
        ' ',
        regex=True,
    )
    s = s.str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip()
    return s.mask(s.eq(''), 'unknown').to_numpy()


def _tag_means(lists, y, smooth=TAG_SMOOTH):
    y = np.asarray(y, dtype=float)
    gmean = float(np.mean(y))
    sums, counts, sq = defaultdict(float), defaultdict(int), defaultdict(float)
    for tags, yi in zip(lists, y):
        for t in tags:
            sums[t] += yi
            sq[t] += yi * yi
            counts[t] += 1
    means = {t: (sums[t] + smooth * gmean) / (counts[t] + smooth) for t in counts}
    tag_var = {}
    for t, c in counts.items():
        tag_var[t] = max(sq[t] / c - (sums[t] / c) ** 2, 0.0) if c > 1 else 0.0
    return means, gmean, counts, tag_var


def _tag_stats(lists, means, gmean, counts, tag_var):
    out = np.empty((len(lists), 5), dtype=float)
    for i, tags in enumerate(lists):
        known = [t for t in tags if t in means]
        if not known:
            out[i] = [gmean, gmean, gmean, 0.0, 0.0]
            continue
        vals = [means[t] for t in known]
        out[i, 0] = np.mean(vals)
        out[i, 1] = np.max(vals)
        out[i, 2] = np.min(vals)
        out[i, 3] = float(np.mean([counts.get(t, 0) for t in known]))
        out[i, 4] = float(np.mean([tag_var.get(t, 0.0) for t in known]))
    return out


def _family_stats(keys, means, gmean, counts):
    out = np.empty((len(keys), 2), dtype=float)
    for i, key in enumerate(keys):
        if key in means:
            out[i, 0] = means[key]
            out[i, 1] = float(counts.get(key, 0))
        else:
            out[i, 0] = gmean
            out[i, 1] = 0.0
    return out


def rank_matrix(df):
    users = df['Users Rated'].to_numpy(float)
    rnk = df['BGG Rank'].to_numpy(float)
    users_c = np.clip(users, 0, None)
    rnk_c = np.clip(rnk, 0, None)
    shrink50 = users_c / (users_c + 50.0)
    weight = users_c / (users_c + 1800.0)
    log_rank = np.log1p(rnk_c)
    return np.column_stack([
        rnk_c,
        log_rank,
        users_c,
        np.log1p(users_c),
        rnk_c / (users_c + 1.0),
        weight,
        rnk_c * shrink50,
        rnk_c * weight,
        log_rank * weight,
    ])


class BoardGameFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, dom_max_features=50, min_df=10, name_svd_components=NAME_SVD_COMPS, mech_svd_components=MECH_SVD_COMPS):
        self.dom_max_features = dom_max_features
        self.min_df = min_df
        self.name_svd_components = name_svd_components
        self.mech_svd_components = mech_svd_components

    def _as_df(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.feature_names_in_)

    def _engineer_num(self, X):
        num = X[NUM_RAW].apply(pd.to_numeric, errors='coerce')
        swap = num['Min Players'] > num['Max Players']
        num.loc[swap, ['Min Players', 'Max Players']] = num.loc[swap, ['Max Players', 'Min Players']].to_numpy()
        year = num['Year Published']
        num['year_cent'] = (year - 2000) / 20
        num['is_old'] = ((year < 1990) & (year != 0) & year.notna()).astype(float)
        num['is_ancient'] = ((year < 1900) & (year != 0) & year.notna()).astype(float)
        num['is_modern'] = (year >= 2015).astype(float)
        num['is_year_future_or_weird'] = ((year < 0) | (year > 2026)).astype(float)
        year_missing = ((year == 0) | year.isna()).astype(float)
        complexity_missing = ((num['Complexity Average'] == 0) | num['Complexity Average'].isna()).astype(float)
        num['owned_per_rated'] = num['Owned Users'] / (num['Users Rated'] + 1)
        num['age_per_complexity'] = num['Min Age'] / (num['Complexity Average'] + 0.1)
        num['time_per_min_player'] = num['Play Time'] / (num['Min Players'].clip(lower=0) + 0.1)
        num['is_filler'] = ((num['Play Time'] > 0) & (num['Play Time'] <= 30)).astype(float)
        num['is_long'] = (num['Play Time'] >= 120).astype(float)
        num['player_group'] = np.select(
            [num['Min Players'] <= 1, num['Min Players'] == 2, num['Min Players'] >= 5],
            [0.0, 1.0, 3.0],
            default=2.0,
        ).astype(float)
        for col in ZERO_AS_NA:
            num.loc[num[col] == 0, col] = np.nan
        num['year_missing'] = year_missing
        num['complexity_missing'] = complexity_missing
        num['is_solo'] = (num['Min Players'] <= 1).astype(float)
        num['is_duel'] = ((num['Min Players'] == 2) & (num['Max Players'] == 2)).astype(float)
        num['is_party'] = (num['Max Players'] >= 7).astype(float)
        num['player_range'] = (num['Max Players'] - num['Min Players']).clip(lower=0)
        num['is_remastered'] = X['is_remastered'].astype(float)
        num['log_users'] = np.log1p(num['Users Rated'].clip(lower=0))
        num['log_owned'] = np.log1p(num['Owned Users'].clip(lower=0))
        num['log_time'] = np.log1p(num['Play Time'].clip(lower=0))
        num['log_rank'] = np.log1p(num['BGG Rank'].clip(lower=0))
        num['rank_per_user'] = num['BGG Rank'] / (num['Users Rated'].clip(lower=0) + 1)
        num['vote_shrink'] = num['Users Rated'].clip(lower=0) / (num['Users Rated'].clip(lower=0) + 50)
        num['vote_weight_1800'] = num['Users Rated'].clip(lower=0) / (num['Users Rated'].clip(lower=0) + 1800)
        num['rank_x_shrink'] = num['BGG Rank'] * num['vote_shrink']
        num['rank_x_weight1800'] = num['BGG Rank'] * num['vote_weight_1800']
        num['log_rank_x_weight1800'] = num['log_rank'] * num['vote_weight_1800']
        num['log_users_per_owned'] = num['log_users'] - num['log_owned']
        num['complexity_x_shrink'] = num['Complexity Average'] * num['vote_shrink']
        num['complexity_sq'] = num['Complexity Average'] ** 2
        num['complexity_x_log_time'] = num['Complexity Average'] * num['log_time']
        num['playtime_per_player'] = num['log_time'] / (num['Max Players'].clip(lower=1) + 0.1)
        mech = X['Mechanics'].fillna('')
        dom = X['Domains'].fillna('')
        num['has_roll_move'] = mech.str.contains('Roll / Spin and Move', regex=False).astype(float)
        num['good_mech_n'] = sum(mech.str.contains(g, regex=False).astype(float) for g in GOOD_MECHS)
        num['has_child_domain'] = dom.str.contains("Children's Games", regex=False).astype(float)
        num['has_strategy'] = dom.str.contains('Strategy Games', regex=False).astype(float)
        num['has_wargame'] = dom.str.contains('Wargames', regex=False).astype(float)
        num['mech_n_tags'] = _norm_tags(X['Mechanics']).map(lambda s: len(_comma_tokens(s)))
        num['dom_n_tags'] = _norm_tags(X['Domains']).map(lambda s: len(_comma_tokens(s)))
        return num

    def _fit_dom(self, X):
        dom_lists = _norm_tags(X['Domains']).map(_comma_tokens)
        dom_counts = Counter(t for tags in dom_lists for t in tags)
        self.dom_keep_ = {lab for lab, cnt in dom_counts.most_common(self.dom_max_features) if cnt >= 1}
        self.mlb_dom_ = MultiLabelBinarizer(classes=sorted(self.dom_keep_))
        self.mlb_dom_.fit(dom_lists.map(lambda xs: [t for t in xs if t in self.dom_keep_]))
        self.dom_cols_ = [f'Domain_{c}' for c in self.mlb_dom_.classes_]
        return dom_lists

    def _dom_frame(self, X):
        dom_lists = _norm_tags(X['Domains']).map(lambda s: [t for t in _comma_tokens(s) if t in self.dom_keep_])
        return pd.DataFrame(self.mlb_dom_.transform(dom_lists), columns=self.dom_cols_, index=X.index)

    def _num_frame(self, X):
        return pd.DataFrame(self.num_imputer_.transform(self._engineer_num(X)), columns=self.num_cols_, index=X.index)

    def fit(self, X, y=None):
        X = self._as_df(X)
        y = np.asarray(y, dtype=float)
        self.feature_names_in_ = np.asarray(X.columns)
        num = self._engineer_num(X)
        self.num_cols_ = list(num.columns)
        self.num_imputer_ = SimpleImputer(strategy='median', keep_empty_features=True)
        self.num_imputer_.fit(num)
        mech_lists = _norm_tags(X['Mechanics']).map(_comma_tokens)
        dom_lists = self._fit_dom(X)
        self.mech_means_, self.mech_gmean_, self.mech_counts_, self.mech_var_ = _tag_means(mech_lists, y)
        self.dom_means_, self.dom_gmean_, self.dom_counts_, self.dom_var_ = _tag_means(dom_lists, y)
        family = _family_keys(X['Name'])
        self.fam_means_, self.fam_gmean_, self.fam_counts_, _ = _tag_means([[k] for k in family], y, smooth=FAMILY_SMOOTH)
        names = X['Name'].fillna('').astype(str)
        self.tfidf_name_ = TfidfVectorizer(**NAME_TFIDF)
        name_tf = self.tfidf_name_.fit_transform(names)
        self.name_ridge_ = Ridge(alpha=10)
        self.name_ridge_.fit(name_tf, y)
        n_svd = min(self.name_svd_components, max(name_tf.shape[1] - 1, 1), max(name_tf.shape[0] - 1, 1))
        self.name_svd_ = TruncatedSVD(n_components=n_svd, random_state=42)
        self.name_svd_.fit(name_tf)
        self.name_svd_cols_ = [f'name_svd_{i}' for i in range(n_svd)]
        self.tfidf_mech_ = TfidfVectorizer(min_df=self.min_df, token_pattern=r'[^ ]+')
        mech_tf = self.tfidf_mech_.fit_transform(X['Mechanics'].fillna('').map(_mech_doc))
        n_mech = min(self.mech_svd_components, max(mech_tf.shape[1] - 1, 1), max(mech_tf.shape[0] - 1, 1))
        self.mech_svd_ = TruncatedSVD(n_components=n_mech, random_state=42)
        self.mech_svd_.fit(mech_tf)
        self.mech_svd_cols_ = [f'mech_svd_{i}' for i in range(n_mech)]
        return self

    def _pack(self, X, mech_te, dom_te, fam_te, name_pred):
        num = self._num_frame(X)
        te = pd.DataFrame(np.hstack([mech_te, dom_te, fam_te]), columns=TE_COLS, index=X.index)
        name_tf = self.tfidf_name_.transform(X['Name'].fillna('').astype(str))
        svd_df = pd.DataFrame(self.name_svd_.transform(name_tf), columns=self.name_svd_cols_, index=X.index)
        mech_tf = self.tfidf_mech_.transform(X['Mechanics'].fillna('').map(_mech_doc))
        mech_df = pd.DataFrame(self.mech_svd_.transform(mech_tf), columns=self.mech_svd_cols_, index=X.index)
        name_df = pd.DataFrame({'oof_name_ridge': name_pred}, index=X.index)
        out = pd.concat([num, te, self._dom_frame(X), mech_df, svd_df, name_df], axis=1)
        self.columns_ = list(out.columns)
        return out.replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()

    def transform(self, X):
        X = self._as_df(X)
        mech_lists = _norm_tags(X['Mechanics']).map(_comma_tokens)
        dom_lists = _norm_tags(X['Domains']).map(_comma_tokens)
        mech_te = _tag_stats(mech_lists, self.mech_means_, self.mech_gmean_, self.mech_counts_, self.mech_var_)
        dom_te = _tag_stats(dom_lists, self.dom_means_, self.dom_gmean_, self.dom_counts_, self.dom_var_)
        fam_te = _family_stats(_family_keys(X['Name']), self.fam_means_, self.fam_gmean_, self.fam_counts_)
        name_pred = self.name_ridge_.predict(self.tfidf_name_.transform(X['Name'].fillna('').astype(str)))
        return self._pack(X, mech_te, dom_te, fam_te, name_pred)

    def fit_transform(self, X, y=None):
        X = self._as_df(X)
        y = np.asarray(y, dtype=float)
        self.fit(X, y)
        mech_lists = _norm_tags(X['Mechanics']).map(_comma_tokens).reset_index(drop=True)
        dom_lists = _norm_tags(X['Domains']).map(_comma_tokens).reset_index(drop=True)
        family = _family_keys(X['Name'])
        names = X['Name'].fillna('').astype(str).to_numpy()
        n = len(X)
        mech_oof = np.zeros((n, 5))
        dom_oof = np.zeros((n, 5))
        fam_oof = np.zeros((n, 2))
        name_oof = np.zeros(n)
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        for tr, va in kf.split(np.arange(n)):
            m_means, m_g, m_c, m_v = _tag_means(mech_lists.iloc[tr], y[tr])
            d_means, d_g, d_c, d_v = _tag_means(dom_lists.iloc[tr], y[tr])
            f_means, f_g, f_c, _ = _tag_means([[k] for k in family[tr]], y[tr], smooth=FAMILY_SMOOTH)
            mech_oof[va] = _tag_stats(mech_lists.iloc[va], m_means, m_g, m_c, m_v)
            dom_oof[va] = _tag_stats(dom_lists.iloc[va], d_means, d_g, d_c, d_v)
            fam_oof[va] = _family_stats(family[va], f_means, f_g, f_c)
            vec = TfidfVectorizer(**NAME_TFIDF)
            ridge = Ridge(alpha=10)
            ridge.fit(vec.fit_transform(names[tr]), y[tr])
            name_oof[va] = ridge.predict(vec.transform(names[va]))
        return self._pack(X, mech_oof, dom_oof, fam_oof, name_oof)


def make_lgbm(n_estimators=450, num_leaves=31, min_child_samples=30, learning_rate=0.05, objective='regression', extra=None):
    params = dict(
        objective=objective,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        verbosity=-1,
        n_jobs=N_JOBS,
    )
    if extra:
        params.update(extra)
    return LGBMRegressor(**params)


def make_resid():
    return make_lgbm(objective='huber', num_leaves=15, extra={'alpha': 1.0})


def make_stage1():
    return make_lgbm(n_estimators=500, num_leaves=31, min_child_samples=20, learning_rate=0.05)


def make_shallow():
    return LGBMRegressor(
        objective='regression',
        n_estimators=40,
        max_depth=3,
        num_leaves=8,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=1.0,
        min_child_samples=20,
        random_state=42,
        verbosity=-1,
        n_jobs=N_JOBS,
    )


def drop_rank(X, names):
    keep = np.array([c not in RANK_DROP for c in names])
    return X[:, keep]


BLEND_TAU = 100.0


def mix_n(a, b, n, tau=BLEND_TAU):
    w = np.asarray(n, dtype=float) / (np.asarray(n, dtype=float) + tau)
    return w * a + (1.0 - w) * b


train['is_remastered'] = train['Name'].str.contains(NAME_RE, case=False, na=False).astype(int)
test['is_remastered'] = test['Name'].str.contains(NAME_RE, case=False, na=False).astype(int)
features = NUM_RAW + ['Name', 'Domains', 'Mechanics', 'is_remastered']
X_df = train[features]
X_test_df = test[features]
rank_X = rank_matrix(train)
rank_X_test = rank_matrix(test)
print('rank features', RANK_COLS)


rank features ['rank', 'log_rank', 'users', 'log_users', 'rank_per_user', 'vote_weight_1800', 'rank_x_shrink50', 'rank_x_weight1800', 'log_rank_x_weight1800']


In [4]:
METHODS = ['inversion', 'stage1', 'inv_residual', 'stage1_residual', 'full', 'blend']
cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof = {name: np.zeros(len(y)) for name in METHODS}
test_sum = {name: np.zeros(len(test)) for name in METHODS}
fold_rows = []


def _oof_slice(n_rows, train_fn, seed):
    pred = np.zeros(n_rows)
    inner = KFold(n_splits=3, shuffle=True, random_state=seed)
    for a, b in inner.split(np.arange(n_rows)):
        pred[b] = train_fn(a, b)
    return pred


for fold, (tr, va) in enumerate(cv.split(y), 1):
    inv_va = invert_rating(rank[tr], y[tr], n_votes[tr], rank[va], n_votes[va], BEST_SPEC)
    inv_te = invert_rating(rank[tr], y[tr], n_votes[tr], rank_test, n_test, BEST_SPEC)
    inv_oof_tr = _oof_slice(
        len(tr),
        lambda a, b: invert_rating(
            rank[tr][a], y[tr][a], n_votes[tr][a], rank[tr][b], n_votes[tr][b], BEST_SPEC,
        ),
        seed=10 + fold,
    )

    stage1 = make_stage1()
    stage1.fit(rank_X[tr], y[tr])
    s1_va = stage1.predict(rank_X[va])
    s1_te = stage1.predict(rank_X_test)
    s1_oof_tr = _oof_slice(
        len(tr),
        lambda a, b, cols=rank_X[tr], target=y[tr]: make_stage1().fit(cols[a], target[a]).predict(cols[b]),
        seed=20 + fold,
    )

    shallow = make_shallow()
    shallow.fit(rank_X[tr], y[tr])
    sh_va = shallow.predict(rank_X[va])
    sh_te = shallow.predict(rank_X_test)
    sh_oof_tr = _oof_slice(
        len(tr),
        lambda a, b, cols=rank_X[tr], target=y[tr]: make_shallow().fit(cols[a], target[a]).predict(cols[b]),
        seed=30 + fold,
    )

    feat = BoardGameFeatures()
    Xtr = feat.fit_transform(X_df.iloc[tr], y[tr])
    Xva = feat.transform(X_df.iloc[va])
    Xte = feat.transform(X_test_df)
    names = feat.columns_
    Xtr_nr, Xva_nr, Xte_nr = drop_rank(Xtr, names), drop_rank(Xva, names), drop_rank(Xte, names)
    if fold == 1:
        print('residual cols', Xtr_nr.shape[1], 'dropped', sorted(RANK_DROP))

    resid_inv = make_resid()
    resid_inv.fit(np.column_stack([Xtr_nr, inv_oof_tr]), y[tr] - inv_oof_tr)
    inv_res_va = inv_va + resid_inv.predict(np.column_stack([Xva_nr, inv_va]))
    inv_res_te = inv_te + resid_inv.predict(np.column_stack([Xte_nr, inv_te]))

    resid_s1 = make_resid()
    resid_s1.fit(np.column_stack([Xtr_nr, s1_oof_tr]), y[tr] - s1_oof_tr)
    s1_res_va = s1_va + resid_s1.predict(np.column_stack([Xva_nr, s1_va]))
    s1_res_te = s1_te + resid_s1.predict(np.column_stack([Xte_nr, s1_te]))

    full = make_lgbm()
    full_tr = np.column_stack([Xtr, inv_oof_tr, s1_oof_tr, sh_oof_tr])
    full_va_x = np.column_stack([Xva, inv_va, s1_va, sh_va])
    full_te_x = np.column_stack([Xte, inv_te, s1_te, sh_te])
    full.fit(full_tr, y[tr])
    full_va = full.predict(full_va_x)
    full_te = full.predict(full_te_x)

    inv_res_va = np.clip(inv_res_va, 1, 10)
    inv_res_te = np.clip(inv_res_te, 1, 10)
    s1_res_va = np.clip(s1_res_va, 1, 10)
    s1_res_te = np.clip(s1_res_te, 1, 10)
    full_va = np.clip(full_va, 1, 10)
    full_te = np.clip(full_te, 1, 10)
    blend_va = np.clip(mix_n(inv_res_va, full_va, n_votes[va]), 1, 10)
    blend_te = np.clip(mix_n(inv_res_te, full_te, n_test), 1, 10)

    preds_va = {
        'inversion': inv_va,
        'stage1': s1_va,
        'inv_residual': inv_res_va,
        'stage1_residual': s1_res_va,
        'full': full_va,
        'blend': blend_va,
    }
    preds_te = {
        'inversion': inv_te,
        'stage1': s1_te,
        'inv_residual': inv_res_te,
        'stage1_residual': s1_res_te,
        'full': full_te,
        'blend': blend_te,
    }
    line = {'fold': fold}
    for name in METHODS:
        oof[name][va] = preds_va[name]
        test_sum[name] += preds_te[name]
        line[name] = _rmse(y[va], preds_va[name])
    fold_rows.append(line)
    print(
        f"fold {fold}  inv={line['inversion']:.4f}  s1={line['stage1']:.4f}  "
        f"inv+res={line['inv_residual']:.4f}  s1+res={line['stage1_residual']:.4f}  full={line['full']:.4f}  blend={line['blend']:.4f}"
    )

print()
print('средний RMSE')
summary = {}
for name in METHODS:
    summary[name] = float(np.mean([row[name] for row in fold_rows]))
    print(f'  {name:18s} {summary[name]:.4f}')
BEST_METHOD = min(summary, key=summary.get)
print('BEST_METHOD', BEST_METHOD)
print('post-hoc mix inv_res/full')
for a in (0.5, 0.7, 0.85, 0.95):
    p = a * oof['inv_residual'] + (1 - a) * oof['full']
    print(f'  w_res={a:.2f}  {_rmse(y, p):.4f}')
print('post-hoc tau n/(n+tau) -> inv_res')
for tau in (50, 100, 300, 1800):
    print(f'  tau={tau:<4}  {_rmse(y, mix_n(oof["inv_residual"], oof["full"], n_votes, tau)):.4f}')


fold 1  inv=0.2433  s1=0.2618  inv+res=0.1938  s1+res=0.2097  full=0.1963
fold 2  inv=0.2493  s1=0.2621  inv+res=0.2073  s1+res=0.2148  full=0.2092
fold 3  inv=0.2461  s1=0.2580  inv+res=0.2094  s1+res=0.2179  full=0.2087
fold 4  inv=0.2216  s1=0.2308  inv+res=0.1869  s1+res=0.1885  full=0.1854
fold 5  inv=0.2513  s1=0.2640  inv+res=0.2071  s1+res=0.2160  full=0.2137

средний RMSE
  inversion          0.2423
  stage1             0.2553
  inv_residual       0.2009
  stage1_residual    0.2094
  full               0.2027
BEST_METHOD inv_residual


In [5]:
test_pred = np.clip(test_sum[BEST_METHOD] / cv.get_n_splits(), 1, 10)
submission = pd.DataFrame({'index': test.index, 'Rating Average': test_pred})
submission.to_csv('submission_residual.csv', index=False)
print(BEST_METHOD, submission['Rating Average'].describe().to_dict())
submission.head()


inv_residual {'count': 5086.0, 'mean': 6.411471210842549, 'std': 0.9153768956654715, 'min': 1.755531310666144, '25%': 5.825689732643092, '50%': 6.44660617563599, '75%': 7.0462543066361905, 'max': 9.160414525412216}
